# 02 — CUAD Ingestion

Downloads the top-20 CUAD contracts from HuggingFace, converts them to PDF,
and uploads them to the Shield AI backend.

**Prerequisites:** run `01_dataset_exploration.ipynb` first so that
`data/cuad_top20_for_ingestion.json` and `data/master_clauses.csv` exist.

## Section 0 — Setup

In [20]:
!pip install fpdf


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [21]:
# %% imports
import json
import os
import re
import string
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from fpdf import FPDF

# %% paths & constants
NOTEBOOK_DIR = Path(".").resolve()
DATA_DIR = NOTEBOOK_DIR / "data"
CUAD_TXTS_DIR = DATA_DIR / "cuad_txts"
CUAD_PDFS_DIR = DATA_DIR / "cuad_pdfs"

CUAD_TXTS_DIR.mkdir(parents=True, exist_ok=True)
CUAD_PDFS_DIR.mkdir(parents=True, exist_ok=True)

BACKEND_URL = "http://localhost:8000"
HF_API_BASE = "https://huggingface.co/api/datasets/theatticusproject/cuad/tree/main/CUAD_v1/full_contract_txt"
HF_RESOLVE_BASE = "https://huggingface.co/datasets/theatticusproject/cuad/resolve/main"

# %% load env
load_dotenv(NOTEBOOK_DIR / ".." / ".env")

print(f"Data dir : {DATA_DIR}")
print(f"TXT dir  : {CUAD_TXTS_DIR}")
print(f"PDF dir  : {CUAD_PDFS_DIR}")
print(f"Backend  : {BACKEND_URL}")

Data dir : /Users/sumithgowrapurashankaramurthy/code/shield-ai/Shield-AI/notebooks/data
TXT dir  : /Users/sumithgowrapurashankaramurthy/code/shield-ai/Shield-AI/notebooks/data/cuad_txts
PDF dir  : /Users/sumithgowrapurashankaramurthy/code/shield-ai/Shield-AI/notebooks/data/cuad_pdfs
Backend  : http://localhost:8000


In [22]:
# %% backend health-check
try:
    resp = requests.get(BACKEND_URL + "/health", timeout=3)
    if resp.ok:
        print("Backend is running:", resp.json())
    else:
        print(f"Backend responded with {resp.status_code} — some upload cells will be skipped.")
    BACKEND_UP = resp.ok
except Exception as e:
    print(f"Backend not running ({e}) — upload cells will use cached results if available.")
    BACKEND_UP = False

Backend is running: {'status': 'ok', 'service': 'shield-ai', 'version': '0.2.0', 'db': 'connected'}


## Section 1 — Load CUAD Top 20

In [23]:
# %% load top-20 list
TOP20_PATH = DATA_DIR / "cuad_top20_for_ingestion.json"

if not TOP20_PATH.exists():
    raise FileNotFoundError(
        f"Missing {TOP20_PATH} — please run 01_dataset_exploration.ipynb first."
    )

with open(TOP20_PATH) as f:
    top20 = json.load(f)

df_top20 = pd.DataFrame(top20)
print(f"Loaded {len(top20)} CUAD contracts for ingestion.")
df_top20[["filename", "inferred_type", "clauses_present", "shield_ai_score"]]

Loaded 20 CUAD contracts for ingestion.


,filename,inferred_type,clauses_present,shield_ai_score
0,TubeMediaCorp_20060310_8-K_EX-10.1_513921_EX-1...,Other / Mixed,40,71.0
1,ScansourceInc_20190822_10-K_EX-10.38_11793958_...,Distribution,40,71.0
2,ENTERTAINMENTGAMINGASIAINC_02_15_2005-EX-10.5-...,Distribution,40,71.0
3,StampscomInc_20001114_10-Q_EX-10.47_2631630_EX...,Other / Mixed,40,71.0
4,"ETELOS,INC_03_09_2004-EX-10.8-DISTRIBUTOR AGRE...",Distribution,40,71.0
5,RandWorldwideInc_20010402_8-KA_EX-10.2_2102464...,Other / Mixed,40,71.0
6,RaeSystemsInc_20001114_10-Q_EX-10.57_2631790_E...,Other / Mixed,40,71.0
7,EUROPEANMICROHOLDINGSINC_03_06_1998-EX-10.6-DI...,Distribution,40,71.0
8,NeoformaInc_19991202_S-1A_EX-10.26_5224521_EX-...,Other / Mixed,40,71.0
9,LeadersonlineInc_20000427_S-1A_EX-10.8_4991089...,Other / Mixed,40,71.0


## Section 2 — Build HuggingFace TXT File Index

In [24]:
# %% fetch HF file listings for Part_I and Part_II
def fetch_hf_listing(part: str) -> list[dict]:
    """Return list of dicts with 'path' and 'rfilename' from HF API."""
    url = f"{HF_API_BASE}/{part}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"WARNING: Could not fetch HF listing for {part}: {e}")
        return []

listing_i  = fetch_hf_listing("Part_I")
listing_ii = fetch_hf_listing("Part_II")

print(f"Part_I  files: {len(listing_i)}")
print(f"Part_II files: {len(listing_ii)}")
print("Sample entry:", listing_i[0] if listing_i else "(none)")

Part_I  files: 100
Part_II files: 100
Sample entry: {'type': 'file', 'oid': '6078257eadbb7ae2797faf22136f3890167a8a0e', 'size': 26346, 'path': 'CUAD_v1/full_contract_txt/Part_I/ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT.txt'}


In [25]:
# %% build filename → download URL mapping
# HF listing entries have a 'path' field like
#   CUAD_v1/full_contract_txt/Part_I/SomeFile.txt
# and a 'rfilename' or 'name' field like SomeFile.txt.

hf_index: dict[str, str] = {}  # stem (no .txt) → full download URL

for entry in listing_i + listing_ii:
    # HF tree API may use 'rfilename', 'name', or 'path'
    path = entry.get("path") or entry.get("rfilename") or ""
    if not path.endswith(".txt"):
        continue
    stem = Path(path).stem  # filename without .txt
    download_url = f"{HF_RESOLVE_BASE}/{path}"
    hf_index[stem] = download_url

print(f"Total indexed TXT files: {len(hf_index)}")
print("Sample keys:", list(hf_index.keys())[:3])

Total indexed TXT files: 200
Sample keys: ['ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT', 'ACCURAYINC_09_01_2010-EX-10.31-DISTRIBUTOR AGREEMENT', 'ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT AGREEMENT']


## Section 3 — Match Top 20 to TXT Files

In [26]:
# %% matching helpers

def normalise(s: str) -> str:
    """Lowercase, strip punctuation, collapse spaces."""
    s = s.lower()
    s = s.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", s).strip()

def fuzzy_match(query_stem: str, index: dict[str, str]) -> tuple[str | None, str]:
    """
    Try:
    1. Exact match on stem.
    2. Case-insensitive exact match on stem.
    3. Normalised substring: check if the key contains ALL tokens from query.
    Returns (matched_stem_or_None, match_type).
    """
    # Strip .pdf from the query filename if present
    q = query_stem
    if q.lower().endswith(".pdf"):
        q = q[:-4]

    # 1. Exact
    if q in index:
        return q, "exact"

    # 2. Case-insensitive
    q_lower = q.lower()
    for k in index:
        if k.lower() == q_lower:
            return k, "case-insensitive"

    # 3. Normalised token overlap (≥ 3 tokens must match)
    q_tokens = set(normalise(q).split())
    # Remove very short / generic tokens
    q_tokens = {t for t in q_tokens if len(t) > 3}
    if q_tokens:
        best_k, best_score = None, 0
        for k in index:
            k_norm = normalise(k)
            k_tokens = set(k_norm.split())
            overlap = len(q_tokens & k_tokens)
            if overlap > best_score:
                best_score = overlap
                best_k = k
        if best_score >= 2:
            return best_k, f"fuzzy ({best_score} tokens)"

    return None, "no match"

# %% run matching
match_rows = []
match_map: dict[str, str | None] = {}  # top20 filename → matched stem

for item in top20:
    fname = item["filename"]
    stem = Path(fname).stem
    matched, mtype = fuzzy_match(stem, hf_index)
    match_map[fname] = matched
    match_rows.append({
        "top20_name": fname[:60] + ("..." if len(fname) > 60 else ""),
        "matched_txt": matched[:60] + ("..." if matched and len(matched) > 60 else "") if matched else "—",
        "match_type": mtype,
    })

df_matches = pd.DataFrame(match_rows)
matched_count = df_matches[df_matches["match_type"] != "no match"].shape[0]
print(f"Matched {matched_count}/{len(top20)} contracts to TXT files.")
df_matches

Matched 6/20 contracts to TXT files.


,top20_name,matched_txt,match_type
0,TubeMediaCorp_20060310_8-K_EX-10.1_513921_EX-1...,—,no match
1,ScansourceInc_20190822_10-K_EX-10.38_11793958_...,ScansourceInc_20190822_10-K_EX-10.38_11793958_...,exact
2,ENTERTAINMENTGAMINGASIAINC_02_15_2005-EX-10.5-...,ENTERTAINMENTGAMINGASIAINC_02_15_2005-EX-10.5-...,exact
3,StampscomInc_20001114_10-Q_EX-10.47_2631630_EX...,—,no match
4,"ETELOS,INC_03_09_2004-EX-10.8-DISTRIBUTOR AGRE...",—,no match
5,RandWorldwideInc_20010402_8-KA_EX-10.2_2102464...,—,no match
6,RaeSystemsInc_20001114_10-Q_EX-10.57_2631790_E...,—,no match
7,EUROPEANMICROHOLDINGSINC_03_06_1998-EX-10.6-DI...,—,no match
8,NeoformaInc_19991202_S-1A_EX-10.26_5224521_EX-...,—,no match
9,LeadersonlineInc_20000427_S-1A_EX-10.8_4991089...,—,no match


## Section 4 — Download TXT Files

In [27]:
# %% download TXT files from HuggingFace

def safe_filename(name: str) -> str:
    """Replace characters unsafe for filenames."""
    return re.sub(r"[^\w\-_.]", "_", name)[:120]

download_results = []  # list of dicts

for item in top20:
    fname = item["filename"]
    matched_stem = match_map.get(fname)

    if matched_stem is None:
        print(f"SKIP (no match): {fname[:70]}")
        download_results.append({"filename": fname, "stem": None, "local_path": None, "status": "no_match"})
        continue

    safe_stem = safe_filename(matched_stem)
    local_path = CUAD_TXTS_DIR / f"{safe_stem}.txt"

    if local_path.exists():
        print(f"CACHED: {local_path.name}")
        download_results.append({"filename": fname, "stem": matched_stem, "local_path": str(local_path), "status": "cached"})
        continue

    url = hf_index[matched_stem]
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        local_path.write_bytes(r.content)
        print(f"OK  ({len(r.content):,} bytes): {local_path.name}")
        download_results.append({"filename": fname, "stem": matched_stem, "local_path": str(local_path), "status": "downloaded"})
    except Exception as e:
        print(f"ERROR downloading {matched_stem}: {e}")
        download_results.append({"filename": fname, "stem": matched_stem, "local_path": None, "status": f"error: {e}"})

    time.sleep(0.5)  # be polite to HuggingFace

df_downloads = pd.DataFrame(download_results)
ok_count = df_downloads[df_downloads["status"].isin(["downloaded", "cached"])].shape[0]
print(f"\nDownloaded/cached: {ok_count}/{len(top20)}")
df_downloads[["filename", "status"]].assign(filename=df_downloads["filename"].str[:60])

SKIP (no match): TubeMediaCorp_20060310_8-K_EX-10.1_513921_EX-10.1_Affiliate Agreement.
CACHED: ScansourceInc_20190822_10-K_EX-10.38_11793958_EX-10.38_Distributor_Agreement1.txt
CACHED: ENTERTAINMENTGAMINGASIAINC_02_15_2005-EX-10.5-DISTRIBUTOR_AGREEMENT.txt
SKIP (no match): StampscomInc_20001114_10-Q_EX-10.47_2631630_EX-10.47_Co-Branding Agree
SKIP (no match): ETELOS,INC_03_09_2004-EX-10.8-DISTRIBUTOR AGREEMENT.PDF
SKIP (no match): RandWorldwideInc_20010402_8-KA_EX-10.2_2102464_EX-10.2_Co-Branding Agr
SKIP (no match): RaeSystemsInc_20001114_10-Q_EX-10.57_2631790_EX-10.57_Co-Branding Agre
SKIP (no match): EUROPEANMICROHOLDINGSINC_03_06_1998-EX-10.6-DISTRIBUTOR AGREEMENT.PDF
SKIP (no match): NeoformaInc_19991202_S-1A_EX-10.26_5224521_EX-10.26_Co-Branding Agreem
SKIP (no match): LeadersonlineInc_20000427_S-1A_EX-10.8_4991089_EX-10.8_Co-Branding Agr
CACHED: NETGEAR_INC_04_21_2003-EX-10.16-AMENDMENT_TO_THE_DISTRIBUTOR_AGREEMENT_BETWEEN_INGRAM_MICRO_AND_NETGEAR.txt
SKIP (no match): LEGACYTEC

,filename,status
0,TubeMediaCorp_20060310_8-K_EX-10.1_513921_EX-1...,no_match
1,ScansourceInc_20190822_10-K_EX-10.38_11793958_...,cached
2,ENTERTAINMENTGAMINGASIAINC_02_15_2005-EX-10.5-...,cached
3,StampscomInc_20001114_10-Q_EX-10.47_2631630_EX...,no_match
4,"ETELOS,INC_03_09_2004-EX-10.8-DISTRIBUTOR AGRE...",no_match
5,RandWorldwideInc_20010402_8-KA_EX-10.2_2102464...,no_match
6,RaeSystemsInc_20001114_10-Q_EX-10.57_2631790_E...,no_match
7,EUROPEANMICROHOLDINGSINC_03_06_1998-EX-10.6-DI...,no_match
8,NeoformaInc_19991202_S-1A_EX-10.26_5224521_EX-...,no_match
9,LeadersonlineInc_20000427_S-1A_EX-10.8_4991089...,no_match


## Section 5 — Convert TXT → PDF

In [28]:
# %% PDF builder — compatible with fpdf 1.x and fpdf2 2.x, handles Unicode

from fpdf import FPDF

def _fpdf_supports_new_x() -> bool:
    import inspect
    return "new_x" in inspect.signature(FPDF.cell).parameters

_NEW_X_SUPPORT = _fpdf_supports_new_x()


def _sanitize(text: str) -> str:
    """Replace common Unicode characters with ASCII equivalents.

    Helvetica in fpdf is a latin-1 core font — any char outside that range
    raises FPDFUnicodeEncodingException. CUAD contracts contain em-dashes,
    smart quotes, etc., so we normalise them before writing to PDF.
    """
    _MAP = {
        "—": "--",    # em-dash
        "–": "-",     # en-dash
        "‘": "'",     # left single quote
        "’": "'",     # right single quote
        "“": '"',     # left double quote
        "”": '"',     # right double quote
        "…": "...",   # ellipsis
        " ": " ",     # non-breaking space
        "•": "*",     # bullet
        "®": "(R)",   # registered trademark
        "©": "(C)",   # copyright
        "™": "(TM)",  # trademark
        "·": "*",     # middle dot
        "‐": "-",     # hyphen
        "‑": "-",     # non-breaking hyphen
        "¶": "",      # paragraph sign
        "§": "S.",    # section sign
    }
    for char, repl in _MAP.items():
        text = text.replace(char, repl)
    # Catch anything else outside latin-1
    return text.encode("latin-1", errors="replace").decode("latin-1")


class ContractPDF(FPDF):
    """PDF generator compatible with fpdf 1.7.x and fpdf2 2.x."""

    def header(self):
        self.set_font("Helvetica", "B", 10)
        title_text = _sanitize(getattr(self, "_contract_title", "") or "")
        if _NEW_X_SUPPORT:
            self.cell(0, 8, title_text[:80], align="C", new_x="LMARGIN", new_y="NEXT")
        else:
            self.cell(0, 8, title_text[:80], 0, 1, "C")
        self.ln(2)

    def footer(self):
        self.set_y(-12)
        self.set_font("Helvetica", "I", 8)
        page_text = f"Page {self.page_no()}"
        if _NEW_X_SUPPORT:
            self.cell(0, 8, page_text, align="C")
        else:
            self.cell(0, 8, page_text, 0, 0, "C")


def txt_to_pdf(txt_path: Path, pdf_path: Path, title: str) -> None:
    """Convert a plain-text contract file to PDF.

    Handles Unicode by sanitising to latin-1 before rendering so that
    the PDF core fonts (Helvetica) don't throw encoding errors on CUAD's
    em-dashes, smart quotes, and special symbols.
    """
    raw = txt_path.read_text(encoding="utf-8", errors="replace")
    text = _sanitize(raw)

    pdf = ContractPDF()
    pdf._contract_title = title[:80]
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Helvetica", size=9)

    CHUNK = 4000
    for i in range(0, len(text), CHUNK):
        chunk = text[i : i + CHUNK]
        pdf.multi_cell(0, 5, chunk)
        if i + CHUNK < len(text):
            pdf.add_page()

    pdf.output(str(pdf_path))


## Section 6 — Upload to Shield AI

In [29]:
# %% upload helper

UPLOAD_RESULTS_PATH = DATA_DIR / "cuad_upload_results.json"

def upload_pdf(pdf_path: Path, actor: str = "user:dataset-ingestion") -> dict:
    """Upload a PDF to Shield AI. Returns result dict."""
    with open(pdf_path, "rb") as fh:
        pdf_bytes = fh.read()

    try:
        r = requests.post(
            f"{BACKEND_URL}/contracts/upload",
            files={"file": (pdf_path.name, pdf_bytes, "application/pdf")},
            data={"actor": actor},
            timeout=60,
        )
        if r.status_code == 409:
            return {"status": "duplicate", "contract_id": None, "raw": r.json()}
        r.raise_for_status()
        data = r.json()
        return {"status": "uploaded", "contract_id": data.get("id") or data.get("contract_id"), "raw": data}
    except requests.exceptions.ConnectionError:
        return {"status": "backend_down", "contract_id": None, "raw": {}}
    except Exception as e:
        return {"status": f"error: {e}", "contract_id": None, "raw": {}}


def poll_contract(contract_id: str, timeout_s: int = 60) -> dict:
    """Poll GET /contracts/{id} until status leaves 'extracted' (or timeout)."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            r = requests.get(f"{BACKEND_URL}/contracts/{contract_id}", timeout=10)
            if r.ok:
                data = r.json()
                status = data.get("status", "")
                if status not in ("uploaded", "processing", "extracted"):
                    return data
            else:
                return {"error": r.status_code}
        except Exception as e:
            return {"error": str(e)}
        time.sleep(5)
    # Timeout — return last known state
    try:
        r = requests.get(f"{BACKEND_URL}/contracts/{contract_id}", timeout=10)
        return r.json() if r.ok else {"error": "timeout"}
    except Exception:
        return {"error": "timeout"}

print("Upload helpers defined.")

Upload helpers defined.


In [30]:
# %% run uploads (skip if backend is down — load cached results instead)

upload_results = []

if not BACKEND_UP:
    print("Backend not running. Loading cached results if available...")
    if UPLOAD_RESULTS_PATH.exists():
        with open(UPLOAD_RESULTS_PATH) as f:
            upload_results = json.load(f)
        print(f"Loaded {len(upload_results)} cached upload results.")
    else:
        print("No cached results found. Run the backend and re-execute this cell.")
else:
    for row in pdf_results:
        if row.get("pdf_path") is None:
            upload_results.append({"filename": row["filename"], "contract_id": None, "status": "no_pdf", "n_clauses": 0, "risk_score": None})
            continue

        pdf_path = Path(row["pdf_path"])
        print(f"Uploading {pdf_path.name}...", end=" ")
        up = upload_pdf(pdf_path)
        contract_id = up.get("contract_id")

        if contract_id:
            print(f"id={contract_id}, polling...", end=" ")
            contract_data = poll_contract(contract_id, timeout_s=60)
            risk = contract_data.get("risk_score")
            n_clauses = len(contract_data.get("clauses", []))
            status = contract_data.get("status", up["status"])
            print(f"status={status}, risk={risk}")
        else:
            contract_data = {}
            risk = None
            n_clauses = 0
            status = up["status"]
            print(status)

        upload_results.append({
            "filename": row["filename"],
            "pdf_path": row.get("pdf_path"),
            "contract_id": contract_id,
            "status": status,
            "n_clauses": n_clauses,
            "risk_score": risk,
        })

        time.sleep(2)

    # Save results
    with open(UPLOAD_RESULTS_PATH, "w") as f:
        json.dump(upload_results, f, indent=2)
    print(f"\nSaved upload results → {UPLOAD_RESULTS_PATH}")

df_uploads = pd.DataFrame(upload_results)
df_uploads.head()


Saved upload results → /Users/sumithgowrapurashankaramurthy/code/shield-ai/Shield-AI/notebooks/data/cuad_upload_results.json


,filename,contract_id,status,n_clauses,risk_score
0,TubeMediaCorp_20060310_8-K_EX-10.1_513921_EX-1...,None,no_pdf,0,None
1,ScansourceInc_20190822_10-K_EX-10.38_11793958_...,None,no_pdf,0,None
2,ENTERTAINMENTGAMINGASIAINC_02_15_2005-EX-10.5-...,None,no_pdf,0,None
3,StampscomInc_20001114_10-Q_EX-10.47_2631630_EX...,None,no_pdf,0,None
4,"ETELOS,INC_03_09_2004-EX-10.8-DISTRIBUTOR AGRE...",None,no_pdf,0,None


## Section 7 — Summary

In [31]:
# %% upload summary counts
import plotly.express as px

if upload_results:
    df_up = pd.DataFrame(upload_results)

    attempted  = len(df_up)
    succeeded  = df_up["contract_id"].notna().sum()
    failed     = df_up["status"].str.startswith("error", na=False).sum()
    duplicates = (df_up["status"] == "duplicate").sum()
    no_pdf     = (df_up["status"] == "no_pdf").sum()

    print("=" * 40)
    print(f"Attempted  : {attempted}")
    print(f"Succeeded  : {succeeded}")
    print(f"Duplicate  : {duplicates}")
    print(f"Failed     : {failed}")
    print(f"No PDF     : {no_pdf}")
    print("=" * 40)

    # Risk score distribution
    risk_data = df_up.dropna(subset=["risk_score"])
    if not risk_data.empty:
        fig = px.histogram(
            risk_data, x="risk_score", nbins=10,
            title="Risk Score Distribution — Uploaded CUAD Contracts",
            labels={"risk_score": "Risk Score"},
            template="plotly_dark",
        )
        fig.show()

    # Contract ID table
    print("\nCUAD filename → Shield AI contract_id:")
    id_table = df_up[["filename", "contract_id", "status", "risk_score"]].copy()
    id_table["filename"] = id_table["filename"].str[:55]
    print(id_table.to_string(index=False))
else:
    print("No upload results available.")

Attempted  : 20
Succeeded  : 0
Duplicate  : 0
Failed     : 0
No PDF     : 20

CUAD filename → Shield AI contract_id:
                                               filename contract_id status risk_score
TubeMediaCorp_20060310_8-K_EX-10.1_513921_EX-10.1_Affil        None no_pdf       None
ScansourceInc_20190822_10-K_EX-10.38_11793958_EX-10.38_        None no_pdf       None
ENTERTAINMENTGAMINGASIAINC_02_15_2005-EX-10.5-DISTRIBUT        None no_pdf       None
StampscomInc_20001114_10-Q_EX-10.47_2631630_EX-10.47_Co        None no_pdf       None
ETELOS,INC_03_09_2004-EX-10.8-DISTRIBUTOR AGREEMENT.PDF        None no_pdf       None
RandWorldwideInc_20010402_8-KA_EX-10.2_2102464_EX-10.2_        None no_pdf       None
RaeSystemsInc_20001114_10-Q_EX-10.57_2631790_EX-10.57_C        None no_pdf       None
EUROPEANMICROHOLDINGSINC_03_06_1998-EX-10.6-DISTRIBUTOR        None no_pdf       None
NeoformaInc_19991202_S-1A_EX-10.26_5224521_EX-10.26_Co-        None no_pdf       None
LeadersonlineInc_200004